In [15]:
import numpy as np
import torch
from torch import nn
from torch_geometric.data import Data
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
import yaml
import os
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    average_precision_score,
    classification_report
)
from tqdm.auto import tqdm

cfg = yaml.safe_load(Path("../../config/model.yaml").read_text())
device = torch.device(
    "cuda" if torch.cuda.is_available() 
    else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(device)

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
set_seed(cfg["seed"])

mps


# Data loading

In [16]:
DATA_DIR = "../../data/processed"

# feats
X_train = np.load(f"{DATA_DIR}/X_train.npy")
X_val   = np.load(f"{DATA_DIR}/X_val.npy")
X_test  = np.load(f"{DATA_DIR}/X_test.npy")

# src-des
pairs_train = np.load(f"{DATA_DIR}/pairs_train.npy", allow_pickle=True)
pairs_val   = np.load(f"{DATA_DIR}/pairs_val.npy", allow_pickle=True)
pairs_test  = np.load(f"{DATA_DIR}/pairs_test.npy", allow_pickle=True)

# labels
y_train = np.load(f"{DATA_DIR}/y_train.npy")
y_val   = np.load(f"{DATA_DIR}/y_val.npy")
y_test  = np.load(f"{DATA_DIR}/y_test.npy")

# convert 1 hot encoded np labels to pytorch labels
y_train_t = torch.tensor(np.argmax(y_train, axis=1), dtype=torch.long)
y_val_t   = torch.tensor(np.argmax(y_val, axis=1), dtype=torch.long)
y_test_t  = torch.tensor(np.argmax(y_test, axis=1), dtype=torch.long)

# verify if shapes of train, test and pair the same

# IP-node mapping

In [17]:
# mapping each ip addrress to nodes
unique_ips = np.unique(np.concatenate([pairs_train[:, 0],pairs_train[:, 1]]))

node_to_id = {
    ip: i
    for i, ip in enumerate(unique_ips)
}

print("Num of nodes:", len(node_to_id))

Num of nodes: 63


In [18]:
# train node ids
src_ids_train = np.array([node_to_id[src] for src in pairs_train[:, 0]])
dst_ids_train = np.array([node_to_id[dst] for dst in pairs_train[:, 1]])
src_ids_train_t = torch.tensor(src_ids_train, dtype=torch.long)
dst_ids_train_t = torch.tensor(dst_ids_train, dtype=torch.long)

# val node ids
src_ids_val = np.array([node_to_id[src] for src in pairs_val[:, 0]])
dst_ids_val = np.array([node_to_id[dst] for dst in pairs_val[:, 1]])
src_ids_val_t = torch.tensor(src_ids_val, dtype=torch.long)
dst_ids_val_t = torch.tensor(dst_ids_val, dtype=torch.long)

# test node ids
src_ids_test = np.array([node_to_id[src] for src in pairs_test[:, 0]])
dst_ids_test = np.array([node_to_id[dst] for dst in pairs_test[:, 1]])
src_ids_test_t = torch.tensor(src_ids_test, dtype=torch.long)
dst_ids_test_t = torch.tensor(dst_ids_test, dtype=torch.long)

# Graph structure

In [19]:
## representation for GCN msg passing
# unique structural connection 
graph_connections = np.unique(np.vstack([src_ids_train, dst_ids_train]), axis=1)
# symmetrise for gcn msg passing
reverse_connections = graph_connections[[1, 0], :]

edge_index = np.unique(
    np.concatenate([graph_connections,reverse_connections],axis=1),
    axis = 1)

edge_index = torch.tensor(edge_index,dtype=torch.long)

## Nodes features

In [20]:
# flatten preprocessed seq feature to aggregate
X_edge_train = X_train.reshape(X_train.shape[0],-1)
X_edge_val = X_val.reshape(X_val.shape[0],-1)
X_edge_test = X_test.reshape(X_test.shape[0],-1)

# build node feat from training set only
num_nodes = len(node_to_id)
num_features = X_edge_train.shape[1]

# 1 feat vector per node
node_features = np.zeros((num_nodes, num_features), dtype=np.float32)
node_counts = np.zeros(num_nodes,dtype=np.float32)

for i in range(len(X_edge_train)):

    src = src_ids_train[i]
    dst = dst_ids_train[i]

    # src and des features
    node_features[src] += X_edge_train[i]
    node_features[dst] += X_edge_train[i]

    node_counts[src] += 1
    node_counts[dst] += 1

# mean feat of all training traffic per ip node
for node in range(num_nodes):
    if node_counts[node] > 0:
        node_features[node] /= node_counts[node]

## Nodes + edges features

In [21]:
# edges feats
edge_features_train = torch.tensor(X_edge_train, dtype=torch.float32)
edge_features_val   = torch.tensor(X_edge_val, dtype=torch.float32)
edge_features_test  = torch.tensor(X_edge_test, dtype=torch.float32)

# GCN to generate node embeddings from node and neighbours
x = torch.tensor(node_features,dtype=torch.float32) # nodes feature
data = Data(x=x, edge_index=edge_index).to(device)

## Model def


In [22]:
class GCN(nn.Module):
    def __init__(self, node_input_dim, hidden_dim, embed_dim, edge_feat_dim, classifier_hidden_dim, dropout, num_classes):

        super().__init__()

        self.conv1 = GCNConv(node_input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, embed_dim)

        self.classifier = nn.Sequential(
            nn.Linear(2 * embed_dim + edge_feat_dim, classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(classifier_hidden_dim, num_classes)
        )

    def forward(self, x, edge_index, src_ids, dst_ids, edge_features):
        h = F.relu(self.conv1(x, edge_index))
        h = self.conv2(h, edge_index)

        # src node embed + dst node embed + edge feat
        edge_attr = torch.cat([h[src_ids], h[dst_ids], edge_features], dim=1)

        return self.classifier(edge_attr)


gcn_cfg = cfg["GCN"]

model = GCN(
    node_input_dim=num_features,
    hidden_dim=gcn_cfg["hidden_dim"],
    embed_dim=gcn_cfg["embed_dim"],
    edge_feat_dim=X_edge_train.shape[1],
    classifier_hidden_dim=gcn_cfg["classifier_hidden_dim"],
    dropout=gcn_cfg["dropout"],
    num_classes=5,
).to(device)

In [23]:
BATCH_SIZE = gcn_cfg["batch_size"]

train_loader = DataLoader(
    TensorDataset(src_ids_train_t, dst_ids_train_t, edge_features_train, y_train_t),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(src_ids_val_t, dst_ids_val_t, edge_features_val, y_val_t),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(src_ids_test_t, dst_ids_test_t, edge_features_test, y_test_t),
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [24]:
model = model.to(device)

# to handle class
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_t.numpy()),
    y=y_train_t.numpy()
)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

# loss fn and optimiser
loss_fn = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=gcn_cfg["lr"],
    weight_decay=gcn_cfg["weight_decay"]
)

In [25]:
def train_gcn(model, data, dataloader, loss_fn, optimizer, device):
    model.train()
    train_loss = 0

    x = data.x
    edge_index = data.edge_index

    for src, dst, edge_feat, y in dataloader:
        src, dst = src.to(device), dst.to(device)
        edge_feat, y = edge_feat.to(device), y.to(device)

        optimizer.zero_grad()

        logits = model(x, edge_index, src, dst, edge_feat)
        loss = loss_fn(logits, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    return train_loss / len(dataloader)

def compute_metrics(probs, preds, labels, num_classes=5):
    # multiclass need to use 1 hot
    labels_onehot = np.eye(num_classes)[labels]

    return {
        "f1": f1_score(labels, preds, average="macro", zero_division=0),
        "precision": precision_score(labels, preds, average="macro", zero_division=0),
        "recall": recall_score(labels, preds, average="macro", zero_division=0),
        "pr_auc": average_precision_score(labels_onehot, probs, average="macro"),
        "report": classification_report(
            labels,
            preds,
            target_names=["Backdoor", "CommInj", "DoS", "Reconn", "normal"],
            digits=4,
            zero_division=0
        )
    }

def evaluate_gcn(model, data, dataloader, loss_fn, device, num_classes=5):
    model.eval()
    val_loss = 0
    all_probs, all_preds, all_labels = [], [], []

    x = data.x.to(device)
    edge_index = data.edge_index.to(device)

    with torch.no_grad():
        for src, dst, edge_feat, y in dataloader:
            src, dst = src.to(device), dst.to(device)
            edge_feat, y = edge_feat.to(device), y.to(device)

            logits = model(x, edge_index, src, dst, edge_feat)
            val_loss += loss_fn(logits, y).item()

            # prob and pred for each batch
            all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
            all_preds.append(logits.argmax(dim=1).cpu().numpy())
            all_labels.append(y.cpu().numpy())

    # flatten into 1 array
    probs = np.concatenate(all_probs)
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)

    metrics = compute_metrics(probs, preds, labels, num_classes)
    metrics["loss"] = val_loss / len(dataloader)

    return metrics

In [ ]:
EPOCHS = gcn_cfg["epochs"]
PATIENCE = gcn_cfg["patience"]

best_pr_auc = 0.0
patience_left = PATIENCE
ckpt_path = "../../models/gcn_best.pt"
os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

pbar = tqdm(range(EPOCHS), desc="Training GCN")

for epoch in pbar:
    train_loss = train_gcn(model, data, train_loader, loss_fn, optimizer, device)
    val_metrics = evaluate_gcn(model, data, val_loader, loss_fn, device)

    pbar.set_postfix(
        train=f"{train_loss:.4f}",
        val=f"{val_metrics['loss']:.4f}",
        f1=f"{val_metrics['f1']:.4f}",
        pr_auc=f"{val_metrics['pr_auc']:.4f}"
    )

    if val_metrics["pr_auc"] > best_pr_auc:
        best_pr_auc = val_metrics["pr_auc"]
        torch.save(model.state_dict(), ckpt_path)
        patience_left = PATIENCE
    else:
        patience_left -= 1

    if patience_left <= 0:
        tqdm.write(f"Early stopping at epoch {epoch + 1}")
        break

Training GCN:   0%|          | 0/30 [00:00<?, ?it/s]

Early stopping at epoch 22


In [27]:
model.load_state_dict(torch.load(ckpt_path, map_location=device))

test_metrics = evaluate_gcn(model,data,test_loader,loss_fn,device)
print(
    f"Test Loss: {test_metrics['loss']:.4f} | "
    f"F1: {test_metrics['f1']:.4f} | "
    f"Precision: {test_metrics['precision']:.4f} | "
    f"Recall: {test_metrics['recall']:.4f} | "
    f"PR-AUC: {test_metrics['pr_auc']:.4f}"
)

print("\nClassification Report:")
print(test_metrics["report"])

Test Loss: 0.0919 | F1: 0.9813 | Precision: 0.9854 | Recall: 0.9780 | PR-AUC: 0.9936

Classification Report:
              precision    recall  f1-score   support

    Backdoor     0.9268    0.9744    0.9500        39
     CommInj     1.0000    0.9804    0.9901        51
         DoS     1.0000    1.0000    1.0000        91
      Reconn     1.0000    0.9355    0.9667        31
      normal     1.0000    1.0000    1.0000     47152

    accuracy                         0.9999     47364
   macro avg     0.9854    0.9780    0.9813     47364
weighted avg     0.9999    0.9999    0.9999     47364

